In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set_style('whitegrid')
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'presentation'

In [2]:
# 1D data
X1d = np.linspace(-1, 1, 10)
y1d = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0, 0])

fig = px.scatter(x=X1d, y=np.zeros(10), color=y1d, title='1D Data', width=600, height=400, color_continuous_scale='Bluered')
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.update_coloraxes(showscale=False)
fig.update_yaxes( range=[-0.1, 0.5])
fig.layout.showlegend = False
fig.show()


In [3]:
# 2D data
X2d = np.square(X1d)
y2d = y1d

fig = make_subplots(rows=1, cols=2, subplot_titles=('1D Data', '2D Data'))

#add traces
fig.add_trace(go.Scatter(x=X1d, y=np.zeros(10), mode='markers', marker=dict(color=y1d, colorscale='Bluered')), row=1, col=1)
fig.add_trace(go.Scatter(x=X1d, y=X2d, mode='markers', marker=dict(color=y2d, colorscale='Bluered')), row=1, col=2)

#update layout
fig.update_layout(width=1000, height=400, title_text='1D vs 2D Data', showlegend=False)

# Rename xaxis
fig.update_xaxes(title_text="X1", row=1, col=1)
fig.update_xaxes(title_text="X1", row=1, col=2)

# Rename yaxis
fig.update_yaxes(title_text="X2", row=1, col=2)

#update marker size
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black', row=1, col=1)
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black', row=1, col=2)

# Update yaxis properties
fig.update_yaxes(range=[-0.1, 1.1], row=1, col=1)
fig.show()

In [4]:
pd.DataFrame({'X1': X1d, 'X2 (X1^2)': X2d, 'y': y2d})

,X1,X2 (X1^2),y
0,-1.000000,1.000000,0
1,-0.777778,0.604938,0
2,-0.555556,0.308642,1
3,-0.333333,0.111111,1
4,-0.111111,0.012346,1
5,0.111111,0.012346,1
6,0.333333,0.111111,1
7,0.555556,0.308642,0
8,0.777778,0.604938,0
9,1.000000,1.000000,0


In [5]:
# Plot linear SVC
from sklearn.svm import SVC
svm = SVC(kernel='linear', C=100)
# Concatenate X1d and X2d
X = np.c_[X1d, X2d]
svm.fit(X, y2d)

svm.intercept_, svm.coef_

(array([2.50059702]), array([[-1.80042989, -8.10193452]]))

In [6]:
fig = px.scatter(x=X1d, y=X2d, color=y2d, title='2D Data', width=600, height=400, color_continuous_scale='Bluered')
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.update_coloraxes(showscale=False)
fig.update_xaxes(title_text="X1")
fig.update_yaxes( title_text="X2", range=[-0.1, 1.1])

# Plot decision boundary
x = np.linspace(-1, 1, 100)     # x1 = -1 to 1
y = -(svm.intercept_[0] + svm.coef_[0][0] * x) / svm.coef_[0][1]       # x2 = -(b + w1*x1) / w2 (The boundary is defined by the line where y=0)
fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='Decision Boundary', showlegend=False))
fig.show()

# Polynomial Kernel

In [7]:
# Polynomial SVC
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# Create polynomial features
poly_svm = Pipeline([('poly', PolynomialFeatures(degree=2,  include_bias=False)), ('svm', SVC(kernel='linear', C=100))])
X = X1d.reshape(-1, 1)
poly_svm.fit(X, y1d)

poly_svm.named_steps['svm'].intercept_, poly_svm.named_steps['svm'].coef_

(array([2.50059702]), array([[-1.80042989, -8.10193452]]))

In [8]:
# Polynomial SVC
poly_svm2 = SVC(kernel='poly', degree=2, C=100)
X = X1d.reshape(-1, 1)
poly_svm2.fit(X, y1d)

SVC(C=100, degree=2, kernel='poly')

# Radial Basis Function Kernel

In [9]:
# RBF SVC
rbf_svm = SVC(kernel='rbf', gamma=1, C=100)
X = X1d.reshape(-1, 1)
rbf_svm.fit(X, y1d)

SVC(C=100, gamma=1)

In [10]:
print('RBF SVC: ', rbf_svm.predict(X))

RBF SVC:  [0 0 1 1 1 1 1 0 0 0]


In [11]:
def gaussian_rbf(x, landmark, gamma):
    return np.exp(-gamma * np.linalg.norm(x - landmark, axis=1)**2)

gamma = 3

x1 = np.linspace(-1, 1, 100).reshape(-1, 1)
x2 = gaussian_rbf(x1, -0.3, gamma)
x3 = gaussian_rbf(x1, 0.3, gamma)

fig = px.scatter(x=X1d, y=np.zeros(10), color=y1d, title='1D Data', width=700, height=500, color_continuous_scale='Bluered')
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.update_coloraxes(showscale=False)
fig.update_yaxes( range=[-0.1, 1.1])

# plot landmarks
fig.add_trace(go.Scatter(x=[-0.3, 0.3], y=[0, 0], mode='markers', marker=dict(color='black', size=10), showlegend=False))

# plot the bell shaped curves
fig.add_trace(go.Scatter(x=x1.reshape(-1), y=x2, mode='lines', name='landmark -0.3'))
fig.add_trace(go.Scatter(x=x1.reshape(-1), y=x3, mode='lines', name='landmark 0.3'))


In [12]:
# plot x2, x3
x1 = np.linspace(-1, 1, 10).reshape(-1, 1)
x2 = gaussian_rbf(x1, -0.3, gamma)
x3 = gaussian_rbf(x1, 0.3, gamma)

fig = px.scatter(x=x2, y=x3, color=y1d, title='2D Data', width=700, height=500, color_continuous_scale='Bluered')
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.update_coloraxes(showscale=False)
fig.update_xaxes(title_text="X2")
fig.update_yaxes( title_text="X3", range=[-0.1, 1.1])


fig.show()

In [13]:
x1 = np.linspace(-1, 1, 10)
x2 = gaussian_rbf(x1.reshape(-1, 1), -0.3, gamma)
x3 = gaussian_rbf(x1.reshape(-1, 1), 0.3, gamma)

pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3, 'y': y2d})

,x1,x2,x3,y
0,-1.000000,0.229925,0.006282,0
1,-0.777778,0.504184,0.030659,0
2,-0.555556,0.822073,0.111255,1
3,-0.333333,0.996672,0.300192,1
4,-0.111111,0.898492,0.602277,1
5,0.111111,0.602277,0.898492,1
6,0.333333,0.300192,0.996672,1
7,0.555556,0.111255,0.822073,0
8,0.777778,0.030659,0.504184,0
9,1.000000,0.006282,0.229925,0


In [14]:
# Train RBF SVC
rbf_svm = SVC(kernel='linear', C=100)
X = np.c_[x2, x3]
rbf_svm.fit(X, y1d)
rbf_svm.predict(X)

array([0, 0, 1, 1, 1, 1, 1, 0, 0, 0])

In [15]:
x = np.linspace(0, 1, 100)
y = -(rbf_svm.intercept_[0] + rbf_svm.coef_[0][0] * x) / rbf_svm.coef_[0][1]
fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='Decision Boundary', showlegend=False))
fig.show()

In [16]:
# xx, yy = np.meshgrid(np.linspace(-0.1, 1.1, 200), np.linspace(-0.1, 1.1, 200).reshape(-1, 1))
# Z = rbf_svm.predict(np.c_[xx.ravel(), yy.ravel()])

# Z = Z.reshape(xx.shape)

# fig.add_trace(go.Contour(x=xx[0], y=yy[:, 0], z=Z, colorscale='Bluered', showscale=False, opacity=0.2, name='Decision Boundary'))

In [17]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=100, noise=0.15, random_state=42)

fig = px.scatter(x=X[:, 0], y=X[:, 1], color=y, title='2D Data', width=700, height=500, color_continuous_scale='Bluered')
fig.update_traces(marker_size=10, marker_line_width=2, marker_line_color='black')
fig.update_coloraxes(showscale=False)
fig.show()

In [18]:
fig = make_subplots(rows=2, cols=2)

for i, gamma, C in ((0, 0.1, 0.001), (1, 0.1, 1000), (2, 5, 0.001), (3, 5, 1000)):
    rbf_svm = SVC(kernel='rbf', gamma=gamma, C=C)
    rbf_svm.fit(X, y)
    xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200).reshape(-1, 1))
    Z = rbf_svm.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    fig.add_trace(go.Contour(x=xx[0], y=yy[:, 0], z=Z, colorscale='Bluered', showscale=False, opacity=0.2, name='Decision Boundary'), row=i//2+1, col=i%2+1)
    fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode='markers', marker=dict(color=y, size=10, colorscale='Bluered'), showlegend=False), row=i//2+1, col=i%2+1)
    fig.update_layout(title_text='Different gamma and C', width=1000, height=800)
    # Titles for each subplot
    fig.update_xaxes(title_text=f"C = {C}, gamma = {gamma}", row=i//2+1, col=i%2+1)

fig.show()